Viewed numerical-stability-and-init.ipynb:1-250
Ran command: `python3 -c "
import json
with open('pytorch/chapter_multilayer-perceptrons/numerical-stability-and-init.ipynb') as f:
    nb = json.load(f)

for i, cell in enumerate(nb['cells']):
    ct = cell['cell_type']
    src = ''.join(cell['source'])
    print(f'=== Cell {i} ({ct}) ===')
    if ct == 'markdown':
        print(src)
    elif ct == 'code':
        print('CODE:\n', src)
"`

本节 notebook [numerical-stability-and-init.ipynb](file:///Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_multilayer-perceptrons/numerical-stability-and-init.ipynb) 主要探讨了深度神经网络中的**数值稳定性（Numerical Stability）**与**参数初始化（Model Initialization）**，阐明了为什么不能随意初始化权重，以及如何通过合理的初始化与激活函数设计避免模型训练失败。

---

### 一、 核心问题：梯度消失与梯度爆炸

在深度神经网络中，前向传播与反向传播依赖大量层间变换的级联。考虑具有 `L` 层的网络，根据链式法则，损失关于第 `l` 层权重 `W⁽ˡ⁾` 的梯度可以表示为多个矩阵与向量的连乘：

```text
反向传播梯度形式：
  ∂o / ∂W⁽ˡ⁾ = M⁽ᴸ⁾ · M⁽ᴸ⁻¹⁾ · ... · M⁽ˡ⁺¹⁾ · v⁽ˡ⁾

其中：
  M⁽ᵏ⁾ = ∂h⁽ᵏ⁾ / ∂h⁽ᵏ⁻¹⁾  (相邻隐藏层间的雅可比矩阵)
  v⁽ˡ⁾ = ∂h⁽ˡ⁾ / ∂W⁽ˡ⁾
```

由于涉及 `L - l` 个矩阵连乘，矩阵特征值的大小直接决定了梯度的稳定性：

1. **梯度消失（Gradient Vanishing）**：
   * **现象**：反向传播回来的梯度极小，导致深层/底层网络参数几乎不更新，模型难以训练收敛。
   * **典型诱因（Sigmoid 函数饱和）**：
     * Sigmoid 函数表达式为 `σ(x) = 1 / (1 + exp(-x))`。
     * 当输入 `|x|` 较大时，函数处于饱和区，导数 `σ'(x) = σ(x)(1 - σ(x))` 迅速趋近于 0（即使在 `x = 0` 时，导数最大也仅为 0.25）。
     * 多层连乘后，浅层梯度呈指数级衰减至 0。
   * **解决方案**：工业实践中普遍采用更稳定的 **ReLU 系列激活函数**（在正区间导数恒为 1）。

2. **梯度爆炸（Gradient Exploding）**：
   * **现象**：矩阵特征值较大时，连乘使得梯度指数级增长，导致数值溢出（`NaN` / `Inf`）或参数大幅振荡，破坏优化算法的稳定性。
   * **示例实验**：Notebook 中使用方差为 1 的高斯随机矩阵 `torch.normal(0, 1, size=(4, 4))` 连续相乘 100 次，矩阵元素数值急剧发散到极其庞大的数量级。

---

### 二、 打破对称性（Symmetry Breaking）

* **对称性问题**：
  * 隐藏单元在结构上具有排列对称性。
  * 如果将某一隐藏层所有权重全初始化为同一个常数 `c`，则该层所有神经元在接收相同输入时，前向计算产生完全相同的激活值；反向传播时根据损失计算出的梯度也完全相同。
* **后果**：无论进行多少轮基于梯度的迭代（如 SGD），所有神经元的权重依然保持完全一致，整层隐藏单元等效于只有**单个神经元**，破坏了神经网络的特征提取与表达能力。
* **解决方式**：
  * 必须使用**随机初始化**打破对称性。
  * 此外，暂退法（Dropout）正则化在训练中也可以辅助打破对称性。

---

### 三、 Xavier 初始化（Glorot & Bengio, 2010）

为了保证数值稳定，核心思想是：**让前向传播中每层输出的方差保持稳定，同时让反向传播中梯度的方差也保持稳定**。

#### 1. 前向方差推导
假设全连接层输入维度为 `n_in`，输出为：
```text
  o_i = ∑_{j=1}^{n_in} w_ij · x_j
```
若权重 `w_ij` 满足零均值、方差 `σ²`，输入 `x_j` 满足零均值、方差 `γ²`，且变量彼此独立，则输出方差为：
```text
  Var[o_i] = n_in · σ² · γ²
```
若要保持输出方差与输入方差一致（`Var[o_i] = γ²`），需要满足：
```text
  n_in · σ² = 1
```

#### 2. 反向梯度方差推导
同理，从输出层往后反向传播梯度时，若输出单元数为 `n_out`，要保持梯度的方差稳定，需要满足：
```text
  n_out · σ² = 1
```

#### 3. Xavier 折中方案
同时严格满足上述两式通常不可行，Xavier 初始化取两者的调和折中，令：
```text
  (1/2) · (n_in + n_out) · σ² = 1
  =>  σ = √(2 / (n_in + n_out))
```

* **正态分布形式**：
  从均值为 0，方差为 `σ² = 2 / (n_in + n_out)` 的正态分布中采样：
  ```text
  W ~ N(0, 2 / (n_in + n_out))
  ```

* **均匀分布形式**：
  由于均匀分布 `U(-a, a)` 的方差为 `a² / 3`，令 `a² / 3 = 2 / (n_in + n_out)`，解得分布上下界：
  ```text
  W ~ U(-√(6 / (n_in + n_out)), √(6 / (n_in + n_out)))
  ```

---

### 四、 核心要点总结

1. **数值不稳定根源**：深层网络反向传播本质是多层雅可比矩阵连乘，特征值过小引发梯度消失，过大引发梯度爆炸。
2. **激活函数选择**：ReLU 相比 Sigmoid 有效缓解了饱和区导致的梯度消失问题。
3. **打破对称性**：严禁用同一常数初始化同一层的所有权重，必须引入随机性。
4. **Xavier 初始化原则**：通过让参数初始方差与输入输出维度关联 `σ = √(2 / (n_in + n_out))`，在训练初期兼顾前向激活与反向梯度的方差平衡。

Viewed 2.7-numerical-stability-and-init.ipynb

以下是针对 notebook [numerical-stability-and-init.ipynb](file:///Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_multilayer-perceptrons/numerical-stability-and-init.ipynb) 课后 4 道习题的详细解答与原理解析。

---

### 习题 1

> **除了多层感知机的排列对称性之外，还能设计出其他神经网络可能会表现出对称性且需要被打破的情况吗？**

**解答：**

在深度学习模型设计中，除了全连接隐藏层的神经元排列对称性外，以下常见架构同样存在严重的对称性问题，必须通过随机初始化或结构扰动予以打破：

1. **卷积神经网络（CNN）中的卷积核/通道对称性**：
   * 在同一个卷积层中通常包含多个输出通道（例如 64 个卷积核）。
   * 如果该层的所有卷积核参数初始化为完全相同（甚至单个卷积核内关于中心旋转对称），那么前向传播时所有输出通道计算出的特征图将完全相同，反向传播时每个卷积核接收到的梯度也完全一致。
   * 这会导致多个通道无法分别提取边缘、纹理、色斑等差异化特征，退化成单一通道的效果。

2. **多头注意力机制（Multi-Head Attention）中的注意力头对称性**：
   * Transformer 的多头注意力由 `h` 个独立的投影矩阵组 `(W_q⁽ⁱ⁾, W_k⁽ⁱ⁾, W_v⁽ⁱ⁾)` 构成。
   * 若所有注意力头的投影权重初始化为相同值，在输入相同的情况下，所有头计算得到的注意力分布和注意力输出完全相同，失去了在多个不同表征子空间中关注上下文信息的意义。

3. **门控循环单元（LSTM / GRU）中的门控参数对称性**：
   * LSTM 中遗忘门、输入门、候选态和输出门分别有独立的权重矩阵。
   * 如果这些门的权重被初始化为相同值，在最初的更新中不同门的行为会产生高度耦合与冗余，难以快速学习到长短期记忆机制。

4. **双分支 / 多分支网络（如 Inception、Siamese 双塔未经解耦的分支）**：
   * 具有对称并行结构的子网络若赋予完全一致的初值，梯度更新方式也将完全同步，无法形成互补特征学习。

---

### 习题 2

> **我们是否可以将线性回归或 softmax 回归中的所有权重参数初始化为相同的值？**

**解答：**

**可以。** 这与多层感知机（MLP）存在本质不同。

1. **线性回归分析**：
   * 线性模型预测为：`y_hat = w_1·x_1 + w_2·x_2 + ... + w_d·x_d + b`。
   * 均方误差关于第 `j` 个权重的梯度为：
     ```text
     ∂L / ∂w_j = (y_hat - y) · x_j
     ```
   * 可以看出，**各个权重的梯度直接正比于对应的输入特征分量 `x_j`**。
   * 只要输入样本的不同特征值不完全相同（`x_1 ≠ x_2`），即使所有权重初始时全设为常数 `c`（甚至全为 0），在第一步反向传播后，各个 `w_j` 获得的梯度也会各不相同，参数立即被分化，不会陷入对称性锁死。
   * 更重要的是，线性回归的目标函数是严格的**凸函数（Convex Function）**，拥有全局唯一最小值，不依赖随机初值来跳出对称鞍点。

2. **Softmax 回归分析**：
   * Softmax 回归也是**无隐藏层的单层线性分类模型**，其优化目标（交叉熵损失）关于分类权重同样是凸优化问题。
   * 即使所有类别的权重全部初始化为 0，模型在初始步给出均匀分布概率 `y_hat_k = 1 / K`。
   * 反向传播时，对于样本真实类别 `c`，其输出梯度为 `1/K - 1`；对于其他类别 `k ≠ c`，梯度为 `1/K`。由不同标签产生的梯度驱动，各个类别的权重向量会迅速产生差异。

**结论**：
* **单层线性模型（线性回归、Softmax 回归）**：不存在隐藏单元的置换对称性，目标函数是凸函数，**初始化为全 0 或相同常数完全可行**。
* **深层/多层神经网络**：具有隐藏层和非线性映射，必须严格使用**随机初始化**打破隐藏层单元间的对称性。

---

### 习题 3

> **在相关资料中查找两个矩阵乘积特征值的解析界。这对确保梯度条件合适有什么启示？**

**解答：**

#### 1. 矩阵乘积特征值与奇异值的解析界
设 `A` 和 `B` 为 `n × n` 矩阵，`||·||₂` 表示矩阵的 2-范数（即最大奇异值 `σ_max`），`ρ(·)` 表示谱半径（最大特征值模长）：

* **次乘性与谱半径界（Gelfand / Ostrowski 相关性质）**：
  ```text
  ρ(A · B) ≤ ||A · B||₂ ≤ ||A||₂ · ||B||₂ = σ_max(A) · σ_max(B)
  ```
* **乘积的最小奇异值下界**：
  ```text
  σ_min(A · B) ≥ σ_min(A) · σ_min(B)
  ```
* 对于对称/正定矩阵，两个矩阵乘积的特征值还受到 Weyl 不等式和 Ostrowski 定理等经典极值特性的约束。

#### 2. 对梯度条件的启示
在深层网络中，输出相对于较早层输入的雅可比矩阵等于每一层局部雅可比矩阵 `M⁽ˡ⁾` 的连乘积：
```text
  M = M⁽ᴸ⁾ · M⁽ᴸ⁻¹⁾ · ... · M⁽¹⁾
```
结合上述界限：
1. **防止梯度爆炸（上界控制）**：
   若各层的最大奇异值 `σ_max(M⁽ˡ⁾) > 1`，则乘积矩阵的范数上限随层数 `L` 呈指数级发散：
   ```text
   ||M||₂ ≤ ∏_{l=1}^{L} σ_max(M⁽ˡ⁾) → ∞
   ```
   **启示**：各层变换的最大奇异值不能大于 1。在实践中，**谱归一化（Spectral Normalization）** 正是通过显式限制每一层权重的最大奇异值 `≤ 1` 来稳定训练。
2. **防止梯度消失（下界控制）**：
   若各层的最小奇异值 `σ_min(M⁽ˡ⁾) < 1` 且接近于 0，连乘后梯度的范数将指数级衰减至 0。
3. **保持等距映射（Isometry）**：
   最理想的梯度传播状态是使每一层的雅可比矩阵近似于**正交矩阵（Orthogonal Matrix）**，即所有奇异值均在 1 附近（`σ_i ≈ 1`），使得向量通过该层变换后长度（方差/能量）既不膨胀也不衰减。这直接催生了**正交初始化（Orthogonal Initialization）** 以及 ResNet 中的**残差连接**（引入恒等映射 `I + F(x)`，让奇异值底线被单位阵维持在 1 附近）。

---

### 习题 4

> **如果我们知道某些项是发散的，我们能在事后修正吗？看看关于按层自适应速率缩放的论文 (You, Gitman, Ginsburg, 2017)。**

**解答：**

**能在事后修正。** 常见的修正思路主要包括梯度裁剪和自适应分层学习率缩放：

#### 1. 传统事后修正：梯度裁剪（Gradient Clipping）
* 当全局梯度范数 `||g||` 超过预设阈值 `θ` 时，进行整体等比例缩减：
  ```text
  g ← g · (θ / ||g||)   (当 ||g|| > θ 时)
  ```
* **局限性**：虽然避免了数值溢出，但使用全局标量缩放无法调节各层之间尺度不均衡的问题。

#### 2. 先进事后修正：LARS（Layer-wise Adaptive Rate Scaling，You et al., 2017）
论文《Large Batch Training of Convolutional Networks》针对超大 Batch Size 训练时不同层梯度更新尺度失衡导致的数值不稳定问题，提出了 **LARS 算法**。

* **问题背景**：
  深层网络在不同层中，权重模长 `||w⁽ˡ⁾||` 与该层梯度模长 `||g⁽ˡ⁾||` 的比值往往相差数个数量级。浅层往往更新过快发生震荡发散，而深层更新极慢。
* **LARS 修正公式**：
  对网络中的每一层 `l`，单独计算一个局部的自适应信任系数（Trust Ratio）：
  ```text
  η⁽ˡ⁾ = η · λ · ( ||w⁽ˡ⁾|| / ( ||g⁽ˡ⁾|| + β · ||w⁽ˡ⁾|| ) )
  ```
  其中：
  * `η` 为全局基础学习率；
  * `λ` 为预设的信任系数（Trust Coefficient，通常约为 0.001）；
  * `β` 为权重衰减（Weight Decay）系数；
  * `||w⁽ˡ⁾||` 和 `||g⁽ˡ⁾||` 分别为第 `l` 层权重和梯度的 L2 范数。
* **事后校正机制**：
  * 当某一层出现梯度发散（`||g⁽ˡ⁾||` 异常巨大）时，该层的局部有效学习率 `η⁽ˡ⁾` 会被自动分母抑制，使该层单步更新步长不超过该层参数自身模长的合理比例。
  * 这种分层自适应修正使得超大批量（如 batch size = 32768）的卷积网络可以在极大学习率下依然稳定收敛，无需因担心局部发散而过分缩减全局学习率。